In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta  # Asegúrate de importar timedelta

# Visitas Tecnicas Brutas

In [ ]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Gestion Traige_wsp_masivo_VTR.csv"

tr = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [ ]:
tr = tr[tr["Periodo"].astype(str).str[:4] == "2026"]

In [ ]:
tr.sample(1)

In [ ]:
conteo_periodo = (
    tr.groupby("Periodo")
      .size()
      .reset_index(name="CANTIDAD")
      .sort_values("Periodo")
)

print("CONTEO POR PERIODO")
print(conteo_periodo)

In [ ]:
# Convertir FECHA DE CREACIÓN a datetime
tr["FECHA DE CREACIÓN"] = pd.to_datetime(tr["FECHA DE CREACIÓN"], errors="coerce")

# Crear campo SEMANA con semana ISO
tr["SEMANA"] = tr["FECHA DE CREACIÓN"].dt.isocalendar().week

In [ ]:
# Convertir fecha
tr["FECHA DE CREACIÓN"] = pd.to_datetime(tr["FECHA DE CREACIÓN"], errors="coerce")

# Crear columnas ISO
iso = tr["FECHA DE CREACIÓN"].dt.isocalendar()
tr["ISO_YEAR"] = iso.year
tr["ISO_WEEK"] = iso.week

tr["YEAR_WEEK"] = (
    tr["ISO_YEAR"].astype(str)
    + "-W"
    + tr["ISO_WEEK"].astype(str).str.zfill(2)
)

In [ ]:
# Obtener semanas únicas ordenadas
semanas = sorted(tr["YEAR_WEEK"].dropna().unique())

# Tomar últimas 4
ultimas_4 = semanas[-5:]

In [ ]:
tr_4s = tr[tr["YEAR_WEEK"].isin(ultimas_4)]

In [ ]:
conteo_4s = (
    tr_4s.groupby("YEAR_WEEK")
         .size()
         .reset_index(name="CANTIDAD")
         .sort_values("YEAR_WEEK")
)

print("\nCONTEO ÚLTIMAS 4 SEMANAS")
print(conteo_4s)

In [ ]:
# Asegurar formato datetime
tr_4s["FECHA DE CREACIÓN"] = pd.to_datetime(tr_4s["FECHA DE CREACIÓN"])

# Filtrar solo el periodo 2026-03
periodo_2026_03 = tr_4s[tr_4s["Periodo"] == "2026-03"]

# Agrupar solo por día (sin hora)
conteo_dias = (
    periodo_2026_03
        .groupby(periodo_2026_03["FECHA DE CREACIÓN"].dt.date)
        .size()
        .reset_index(name="CANTIDAD")
        .sort_values("FECHA DE CREACIÓN")
)

print("\nCONTEO DIARIO - PERIODO 2026-03")
print(conteo_dias)

# Bases Asignaciones 

In [2]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Asignacion_CLARO.csv"

ASIGCLARO = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [3]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Asignacion_VTR.csv"

AGIGVTR = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [ ]:
# Lista de palabras clave
keywords_casillero = r"CJUG|NJUG|EDES|NJUGCALL"

for df_base in [ASIGCLARO, AGIGVTR]:
    
    df_base["casillero"] = df_base["casillero"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["casillero"]
        .str.upper()
        .str.contains(keywords_casillero, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa de masivo)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [ ]:
for df_base in [ASIGCLARO, AGIGVTR]:
    
    # Asegurar tipo texto
    df_base["Tipo_Asignacion"] = df_base["Tipo_Asignacion"].astype(str)
    
    # Crear campo Masivo
    df_base["Masivo"] = (
        df_base["Tipo_Asignacion"]
        .str.contains("Masivo", case=False, na=False)
        .astype(int)
    )
    
    # Crear campo LOGICA (inverso)
    df_base["LOGICA"] = (1 - df_base["Masivo"]).astype(int)

In [ ]:
AGIGVTR.sample(1)

In [ ]:
# Convertir a datetime
ASIGCLARO["fecha"] = pd.to_datetime(ASIGCLARO["fecha"], errors="coerce")
AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")

In [ ]:
import pandas as pd

# Fecha de hoy
hoy = pd.Timestamp.today().normalize()

# Excluir registros del día actual
ASIGCLARO = ASIGCLARO[ASIGCLARO["fecha"] < hoy]
AGIGVTR = AGIGVTR[AGIGVTR["fecha"] < hoy]

# Crear columna MES (formato YYYY-MM)
ASIGCLARO["MES"] = ASIGCLARO["fecha"].dt.to_period("M")
AGIGVTR["MES"] = AGIGVTR["fecha"].dt.to_period("M")

# Conteo mensual
claro_mes = ASIGCLARO.groupby("MES").size()
vtr_mes = AGIGVTR.groupby("MES").size()

# Unir y sumar
conteo_mes = pd.concat([claro_mes, vtr_mes], axis=1)
conteo_mes.columns = ["CLARO", "VTR"]
conteo_mes = conteo_mes.fillna(0)

conteo_mes["TOTAL"] = conteo_mes["CLARO"] + conteo_mes["VTR"]

print("CONTEO MENSUAL")
print(conteo_mes.sort_index())

In [ ]:
import pandas as pd

# Asegurar formato fecha
ASIGCLARO["fecha"] = pd.to_datetime(ASIGCLARO["fecha"], errors="coerce")
AGIGVTR["fecha"] = pd.to_datetime(AGIGVTR["fecha"], errors="coerce")

# Crear año y semana ISO
for df_base in [ASIGCLARO, AGIGVTR]:
    iso = df_base["fecha"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week
    df_base["YEAR_WEEK"] = df_base["ISO_YEAR"].astype(str) + "-W" + df_base["ISO_WEEK"].astype(str).str.zfill(2)

In [ ]:
# Unimos semanas de ambas bases
semanas_disponibles = pd.concat([
    ASIGCLARO["YEAR_WEEK"],
    AGIGVTR["YEAR_WEEK"]
]).dropna().unique()

# Ordenar correctamente
semanas_disponibles = sorted(semanas_disponibles)

# Tomar últimas 3
ultimas_4 = semanas_disponibles[-5:]

In [ ]:
claro_4s = ASIGCLARO[ASIGCLARO["YEAR_WEEK"].isin(ultimas_4)]
vtr_4s = AGIGVTR[AGIGVTR["YEAR_WEEK"].isin(ultimas_4)]

In [ ]:
claro_sem = claro_4s.groupby("YEAR_WEEK").size()
vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()

conteo_4s = pd.concat([claro_sem, vtr_sem], axis=1)
conteo_4s.columns = ["CLARO", "VTR"]
conteo_4s = conteo_4s.fillna(0)

conteo_4s["TOTAL"] = conteo_4s["CLARO"] + conteo_4s["VTR"]

print("\nCONTEO ÚLTIMAS 3 SEMANAS")
print(conteo_4s.sort_index())

In [ ]:
# Asegurar marca
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# Resumen base por Periodo y Marca
resumen = (
    base_asig
    .groupby(["Periodo", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Calcular totales por Periodo (CLARO + VTR)
totales_periodo = (
    resumen
    .groupby("Periodo")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Hacer merge para agregar los totales a cada fila
resumen_final = resumen.merge(totales_periodo, on="Periodo", how="left")

# Ordenar
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

print(resumen_final)

In [ ]:
# Asegurar marca
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# Resumen base por Periodo y Marca
resumen = (
    base_asig
    .groupby(["YEAR_WEEK", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Calcular totales por Periodo (CLARO + VTR)
totales_periodo = (
    resumen
    .groupby("YEAR_WEEK")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Hacer merge para agregar los totales a cada fila
resumen_final = resumen.merge(totales_periodo, on="YEAR_WEEK", how="left")

# Ordenar
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(resumen_final)

In [ ]:
import pandas as pd

# ==============================
# ASEGURAR MARCA
# ==============================
ASIGCLARO["MARCA"] = "CLARO"
AGIGVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_asig = pd.concat([ASIGCLARO, AGIGVTR], ignore_index=True)

# ==============================
# ASEGURAR FECHA EN FORMATO DATETIME
# ==============================
base_asig["fecha"] = pd.to_datetime(base_asig["fecha"])

# ==============================
# FILTRAR PERIODO 2026-03
# ==============================
base_2026_03 = base_asig[base_asig["Periodo"] == "2026-03"].copy()

# Crear columna solo fecha (sin hora)
base_2026_03["FECHA_DIA"] = base_2026_03["fecha"].dt.date

# ==============================
# RESUMEN POR DIA Y MARCA
# ==============================
resumen = (
    base_2026_03
    .groupby(["FECHA_DIA", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR DIA (CLARO + VTR)
# ==============================
totales_dia = (
    resumen
    .groupby("FECHA_DIA")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_dia,
    on="FECHA_DIA",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["FECHA_DIA", "MARCA"])

print("\n===== RESUMEN DIARIO ASIGNACIONES - PERIODO 2026-03 =====")
print(resumen_final)

# Bases Recorrido

In [4]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Recorrido_VTR.csv"

recoVTR = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [5]:
ruta = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Triage Documentos Operaciones\Recorrido_CLARO.csv"

recoCLARO = pd.read_csv(ruta, sep=";", encoding="utf-8")

In [ ]:
# Lista de palabras clave
keywords_casillero = r"CJUG|NJUG|EDES|NJUGCALL"

for df_base in [recoVTR]:
    
    df_base["CASILLERO"] = df_base["CASILLERO"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["CASILLERO"]
        .str.upper()
        .str.contains(keywords_casillero, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa de masivo)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [ ]:
recoVTR["LOGICA"].value_counts()

In [ ]:
for df_base in [recoCLARO]:
    
    df_base["PUESTO_TRABAJO"] = df_base["PUESTO_TRABAJO"].astype(str)
    
    # Crear LOGICA_MASIVO
    df_base["LOGICA_MASIVO"] = (
        df_base["PUESTO_TRABAJO"]
        .str.contains("Masivo", case=False, na=False)
        .astype(int)
    )
    
    # Crear LOGICA (inversa)
    df_base["LOGICA"] = (df_base["LOGICA_MASIVO"] == 0).astype(int)

In [ ]:
recoCLARO["LOGICA"].value_counts()

In [ ]:
# Convertir a datetime
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")

# Extraer YYYY-MM-DD
recoVTR["FECHA_INICIO"] = recoVTR["FECHA_INICIO"].dt.date

In [ ]:
# Convertir a datetime
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Extraer YYYY-MM-DD
recoCLARO["FECHA_INICIO"] = recoCLARO["FECHA_INICIO"].dt.date

In [ ]:
# Convertir FECHA_INICIO a datetime
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")

# Crear campo SEMANA con semana ISO
recoVTR["SEMANA"] = recoVTR["FECHA_INICIO"].dt.isocalendar().week

In [ ]:
# Convertir FECHA_INICIO a datetime
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Crear campo SEMANA con semana ISO
recoCLARO["SEMANA"] = recoCLARO["FECHA_INICIO"].dt.isocalendar().week

In [ ]:
# Ajustar las opciones para mostrar todas las columnas
pd.set_option('display.max_columns', None)

In [ ]:
recoVTR = recoVTR[
    ~(
        recoVTR["GESTION"].str.contains(
            "Orden de reparacion ya cancelada",
            case=False,
            na=False
        )
        |
        recoVTR["CASILLERO"].str.contains(
            "CP05",
            case=False,
            na=False
        )
    )
]

In [ ]:
recoCLARO = recoCLARO[
    ~(
        recoCLARO["GESTION"].str.contains(
            "no corresponde gesti",
            case=False,
            na=False
        ))
]

In [ ]:
# Convertir a fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Tomar el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# Conteo por periodo
vtr_periodo = recoVTR["Periodo"].value_counts()
claro_periodo = recoCLARO["Periodo"].value_counts()

# Unir resultados
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0)

# Total
conteo_periodo["TOTAL"] = conteo_periodo["VTR"] + conteo_periodo["CLARO"]

print("\nCONTEO RECORRIDO")
print(conteo_periodo.sort_index())

In [ ]:
# Convertir a fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# Agregar marca
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# Unir bases
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# Resumen por Periodo y Marca
resumen = (
    base_reco
    .groupby(["Periodo", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Totales por Periodo
totales_periodo = (
    resumen
    .groupby("Periodo", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Agregar totales
resumen_final = resumen.merge(
    totales_periodo,
    on="Periodo",
    how="left"
)

# Ordenar
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

print("\nCONTEO RECORRIDO POR LOGICA")
print(resumen_final)

In [ ]:
for df_base in [recoVTR, recoCLARO]:

    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()

    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    # Crear YEAR_WEEK solo si no es nulo
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype("Int64").astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype("Int64").astype(str).str.zfill(2)
    )

In [ ]:
# =========================
# LIMPIEZA Y ÚLTIMO REGISTRO
# =========================

# Convertir fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# PREPARAR BASE
# =========================

recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# Unir bases
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# RESUMEN POR SEMANA
# =========================

resumen = (
    base_reco
    .groupby(["YEAR_WEEK", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# Totales por semana
totales_periodo = (
    resumen
    .groupby("YEAR_WEEK", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# Merge totales
resumen_final = resumen.merge(
    totales_periodo,
    on="YEAR_WEEK",
    how="left"
)

# Ordenar
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

# =========================
# 🔥 ÚLTIMAS SEMANAS
# =========================

semanas = (
    resumen_final["YEAR_WEEK"]
    .dropna()
    .sort_values()
    .unique()
)

ultimas_4 = semanas[-6:]

resumen_ultimas_4 = resumen_final[
    resumen_final["YEAR_WEEK"].isin(ultimas_4)
]

print("\n===== RECO - ÚLTIMAS SEMANAS =====")
print(resumen_ultimas_4)

In [ ]:
for df_base in [recoVTR, recoCLARO]:

    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()

    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    # Crear YEAR_WEEK solo si no es nulo
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype("Int64").astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype("Int64").astype(str).str.zfill(2)
    )

In [ ]:
semanas = sorted(
    pd.concat([
        recoVTR["YEAR_WEEK"],
        recoCLARO["YEAR_WEEK"]
    ])
    .dropna()
    .loc[lambda x: ~x.str.contains("<NA>")]  # elimina basura
    .unique()
)

ultimas_4 = semanas[-20:]

In [ ]:
# =========================
# LIMPIEZA Y ÚLTIMO REGISTRO
# =========================

# Convertir fecha
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# Eliminar registros sin identificador o fecha
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# Ordenar por fecha
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# Quedarse con el último registro por orden/folio
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# FILTRAR ÚLTIMAS SEMANAS
# =========================

vtr_4s = recoVTR[recoVTR["YEAR_WEEK"].isin(ultimas_4)]
claro_4s = recoCLARO[recoCLARO["YEAR_WEEK"].isin(ultimas_4)]

# Conteo individual
vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()
claro_sem = claro_4s.groupby("YEAR_WEEK").size()

# Unir
conteo_sem = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_sem.columns = ["VTR", "CLARO"]
conteo_sem = conteo_sem.fillna(0).astype(int)

# Total
conteo_sem["TOTAL"] = (
    conteo_sem["VTR"] + conteo_sem["CLARO"]
).astype(int)

print("\nCONTEO RECORRIDO ÚLTIMAS 4 SEMANAS")
print(conteo_sem.sort_index())

In [ ]:
import pandas as pd

# =========================
# ASEGURAR FORMATO FECHA
# =========================
recoVTR["FECHA_INICIO"] = pd.to_datetime(recoVTR["FECHA_INICIO"], errors="coerce")
recoCLARO["FECHA_INICIO"] = pd.to_datetime(recoCLARO["FECHA_INICIO"], errors="coerce")

# =========================
# ELIMINAR REGISTROS INVALIDOS
# =========================
recoVTR = recoVTR.dropna(subset=["NMRO_ORDEN", "FECHA_INICIO"])
recoCLARO = recoCLARO.dropna(subset=["FOLIO", "FECHA_INICIO"])

# =========================
# ORDENAR POR FECHA
# =========================
recoVTR = recoVTR.sort_values("FECHA_INICIO")
recoCLARO = recoCLARO.sort_values("FECHA_INICIO")

# =========================
# TOMAR ULTIMO REGISTRO POR ORDEN
# =========================
recoVTR = recoVTR.drop_duplicates(subset="NMRO_ORDEN", keep="last")
recoCLARO = recoCLARO.drop_duplicates(subset="FOLIO", keep="last")

# =========================
# ASEGURAR MARCA
# =========================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# =========================
# UNIR BASES
# =========================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# CREAR COLUMNA DIA
# =========================
base_reco["DIA"] = base_reco["FECHA_INICIO"].dt.date

# =========================
# FILTRAR SOLO PERIODO 2026-03
# =========================
base_reco = base_reco[base_reco["Periodo"] == "2026-03"]

# =========================
# RESUMEN POR DIA Y MARCA
# =========================
resumen = (
    base_reco
    .groupby(["DIA", "MARCA"], observed=True)
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# =========================
# TOTALES POR DIA
# =========================
totales_dia = (
    resumen
    .groupby("DIA", observed=True)
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# =========================
# MERGE TOTALES
# =========================
resumen_final = resumen.merge(
    totales_dia,
    on="DIA",
    how="left"
)

# =========================
# ORDENAR
# =========================
resumen_final = resumen_final.sort_values(["DIA", "MARCA"])

print("\nCONTEO RECORRIDO POR LOGICA - DIARIO PERIODO 2026-03")
print(resumen_final)

# Bases Contactabilidad

In [ ]:
# Eliminar registros que contengan "sin contacto"
recoVTR = recoVTR[
    ~recoVTR["GESTION"].str.contains("sin contacto", case=False, na=False)
].copy()

recoCLARO = recoCLARO[
    ~recoCLARO["GESTION"].str.contains("sin contacto", case=False, na=False)
].copy()

In [ ]:
# Conteo individual
vtr_periodo = recoVTR.groupby("Periodo").size()
claro_periodo = recoCLARO.groupby("Periodo").size()

# Unir
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0).astype(int)

# Total
conteo_periodo["TOTAL"] = (
    conteo_periodo["VTR"] + conteo_periodo["CLARO"]
).astype(int)

print("\n===== Contactabilidad POR PERIODO =====")
print(conteo_periodo.sort_index())

In [ ]:
# Asegurar fecha
for df_base in [recoVTR, recoCLARO]:
    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )
    
    iso = df_base["FECHA_INICIO"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week
    
    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype(str).str.zfill(2)
    )

In [ ]:
# Conteo individual
vtr_sem = recoVTR.groupby("YEAR_WEEK").size()
claro_sem = recoCLARO.groupby("YEAR_WEEK").size()

# Unir
conteo_semana = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_semana.columns = ["VTR", "CLARO"]
conteo_semana = conteo_semana.fillna(0).astype(int)

# Total
conteo_semana["TOTAL"] = (
    conteo_semana["VTR"] + conteo_semana["CLARO"]
).astype(int)

print("\n===== Contactabilidad POR SEMANA =====")
print(conteo_semana.sort_index())

In [ ]:
import pandas as pd

# ==============================
# ASEGURAR MARCA
# ==============================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# ==============================
# RESUMEN POR PERIODO Y MARCA
# ==============================
resumen = (
    base_reco
    .groupby(["Periodo", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR PERIODO (CLARO + VTR)
# ==============================
totales_periodo = (
    resumen
    .groupby("Periodo")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_periodo,
    on="Periodo",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["Periodo", "MARCA"])

# ==============================
# PRINT FINAL
# ==============================
print("\n===== CONTACTABILIDAD - RESUMEN POR PERIODO =====")
print(resumen_final)

In [ ]:
# ==============================
# ASEGURAR MARCA
# ==============================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# ==============================
# UNIR BASES
# ==============================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# ==============================
# RESUMEN POR SEMANA Y MARCA
# ==============================
resumen = (
    base_reco
    .groupby(["YEAR_WEEK", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# ==============================
# TOTALES POR SEMANA (CLARO + VTR)
# ==============================
totales_semana = (
    resumen
    .groupby("YEAR_WEEK")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# ==============================
# MERGE TOTALES
# ==============================
resumen_final = resumen.merge(
    totales_semana,
    on="YEAR_WEEK",
    how="left"
)

# ==============================
# ORDENAR
# ==============================
resumen_final = resumen_final.sort_values(["YEAR_WEEK", "MARCA"])

# ==============================
# FILTRAR ULTIMAS 5 SEMANAS
# ==============================
ultimas_5 = sorted(resumen_final["YEAR_WEEK"].unique())[-6:]

resumen_final_ult5 = (
    resumen_final[
        resumen_final["YEAR_WEEK"].isin(ultimas_5)
    ]
    .sort_values(["YEAR_WEEK", "MARCA"])
)

# ==============================
# PRINT FINAL
# ==============================
print("\n===== CONTACTABILIDAD - RESUMEN ULTIMAS 5 SEMANAS =====")
print(resumen_final_ult5)

In [ ]:
# =========================
# ASEGURAR MARCA
# =========================
recoCLARO["MARCA"] = "CLARO"
recoVTR["MARCA"] = "VTR"

# =========================
# UNIR BASES
# =========================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

# =========================
# ASEGURAR FORMATO FECHA
# =========================
base_reco["FECHA_INICIO"] = pd.to_datetime(base_reco["FECHA_INICIO"])

# Crear columna DIA
base_reco["DIA"] = base_reco["FECHA_INICIO"].dt.date

# =========================
# FILTRAR SOLO PERIODO 2026-03
# =========================
base_reco = base_reco[base_reco["Periodo"] == "2026-03"]

# =========================
# RESUMEN POR DIA Y MARCA
# =========================
resumen = (
    base_reco
    .groupby(["DIA", "MARCA"])
    .agg(
        MASIVAS=("LOGICA_MASIVO", "sum"),
        LOGICAS=("LOGICA", "sum"),
        TOTAL=("LOGICA_MASIVO", "count")
    )
    .astype(int)
    .reset_index()
)

# =========================
# TOTALES POR DIA
# =========================
totales_dia = (
    resumen
    .groupby("DIA")
    .agg(
        TOTAL_MASIVO=("MASIVAS", "sum"),
        TOTAL_LOGICA=("LOGICAS", "sum")
    )
    .reset_index()
)

# =========================
# MERGE TOTALES
# =========================
resumen_final = resumen.merge(
    totales_dia,
    on="DIA",
    how="left"
)

# =========================
# ORDENAR
# =========================
resumen_final = resumen_final.sort_values(["DIA", "MARCA"])

print("\nCONTEO CONTACTADO POR LOGICA - DIARIO PERIODO 2026-03")
print(resumen_final)

# Cancelaciones

In [ ]:
recoVTR = recoVTR[
    recoVTR["Resultado Final Orden"]
        .str.replace(r"\s+", " ", regex=True)
        .str.contains("cancelado", case=False, na=False)
].copy()

recoCLARO = recoCLARO[
    recoCLARO["Resultado Final Orden"]
        .str.replace(r"\s+", " ", regex=True)
        .str.contains("cancelado", case=False, na=False)
].copy()

In [ ]:
# Conteo individual
vtr_periodo = recoVTR.groupby("Periodo").size()
claro_periodo = recoCLARO.groupby("Periodo").size()

# Unir
conteo_periodo = pd.concat([vtr_periodo, claro_periodo], axis=1)
conteo_periodo.columns = ["VTR", "CLARO"]
conteo_periodo = conteo_periodo.fillna(0).astype(int)

# Total
conteo_periodo["TOTAL"] = conteo_periodo["VTR"] + conteo_periodo["CLARO"]

print("\n===== CANCELADOS - CONTEO POR PERIODO =====")
print(conteo_periodo.sort_index())

In [ ]:
# Crear YEAR_WEEK si no existe
for df_base in [recoVTR, recoCLARO]:
    df_base["FECHA_INICIO"] = pd.to_datetime(
        df_base["FECHA_INICIO"],
        errors="coerce"
    )

    iso = df_base["FECHA_INICIO"].dt.isocalendar()
    df_base["ISO_YEAR"] = iso.year
    df_base["ISO_WEEK"] = iso.week

    df_base["YEAR_WEEK"] = (
        df_base["ISO_YEAR"].astype(str)
        + "-W"
        + df_base["ISO_WEEK"].astype(str).str.zfill(2)
    )

In [ ]:
# Obtener semanas válidas
semanas = sorted(
    pd.concat([
        recoVTR["YEAR_WEEK"],
        recoCLARO["YEAR_WEEK"]
    ])
    .dropna()
    .unique()
)

ultimas_4 = semanas[-4:]

In [ ]:
vtr_4s = recoVTR[recoVTR["YEAR_WEEK"].isin(ultimas_4)]
claro_4s = recoCLARO[recoCLARO["YEAR_WEEK"].isin(ultimas_4)]

vtr_sem = vtr_4s.groupby("YEAR_WEEK").size()
claro_sem = claro_4s.groupby("YEAR_WEEK").size()

conteo_semana = pd.concat([vtr_sem, claro_sem], axis=1)
conteo_semana.columns = ["VTR", "CLARO"]
conteo_semana = conteo_semana.fillna(0).astype(int)

conteo_semana["TOTAL"] = (
    conteo_semana["VTR"] + conteo_semana["CLARO"]
)

print("\n===== CANCELADOS - ÚLTIMAS 4 SEMANAS =====")
print(conteo_semana.sort_index())

# Por Logica

In [ ]:
recoCLARO.rename(columns={"casillero": "CASILLERO"}, inplace=True)

In [ ]:
# Unir ambas bases
base_total = pd.concat([recoVTR, recoCLARO], ignore_index=True)

suma_periodo = (
    base_total
    .groupby("Periodo")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

print("\n===== SUMA POR PERIODO LOGICAS =====")
print(suma_periodo)

In [ ]:
suma_semana = (
    base_total
    .groupby("YEAR_WEEK")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

# Tomar solo las últimas 5 semanas
suma_semana = suma_semana.tail(30)

print("\n===== SUMA POR SEMANA - ULTIMAS 5 SEMANAS =====")
print(suma_semana)

In [ ]:
# Filtrar solo periodo 2026-03
base_filtrada = base_total[base_total["Periodo"] == "2026-03"]

# Asegurar formato fecha
base_filtrada["FECHA_INICIO"] = pd.to_datetime(base_filtrada["FECHA_INICIO"])

# Crear columna DIA
base_filtrada["DIA"] = base_filtrada["FECHA_INICIO"].dt.date

# Suma por día
suma_dia = (
    base_filtrada
    .groupby("DIA")[["LOGICA_MASIVO", "LOGICA"]]
    .sum()
    .astype(int)
    .sort_index()
)

print("\n===== SUMA POR DIA - PERIODO 2026-03 =====")
print(suma_dia)

# NPS Triage 

In [6]:
# Ajustar las opciones para mostrar todas las columnas
pd.set_option('display.max_columns', None)

In [7]:
# =========================
# JOIN PARA AGREGAR rut_cliente
# =========================
recoVTR = recoVTR.merge(
    AGIGVTR[['numero_solicitud', 'rut_cliente']],
    left_on='NMRO_ORDEN',
    right_on='numero_solicitud',
    how='left'
)

# Opcional: eliminar la columna auxiliar
recoVTR = recoVTR.drop(columns=['numero_solicitud'])

In [8]:
recoVTR['MARCA'] = 'VTR'

In [9]:
recoCLARO.sample(1)

,ID,USER,FOLIO,TIPO_DE_ORDEN,CRM,GESTION,PUESTO_TRABAJO,CASUISTICA,SERVICIO,PRUEBAS,SOLUCION,OBSERVACION,FECHA_INICIO,FECHA_GESTION,Periodo,Resultado Final Orden,FECHA_INICIO2,Flag Can,Flag MV,Sin Contacto
50611,180056,1912122T4,1-256611130504,Actividad remota,ANDES,Cliente acepta script,Masivo Finalizado,Ninguno,Masivo,No,Ninguno,@claroTR AS 1-256611130504 CL INDICA LOS SERVI...,2025-04-13T16:16,"2025-04-13 16:20:30,000",2025-04,Cancelado,"45760,680902777778",1,0,0


In [10]:
AGIGVTR.sample(1)

,estado,ID_SS,numero_solicitud,cuidad,fecha_agendamiento,hora,casillero,puesto_de_trabajo,rut_cliente,CNC,fecha,Periodo,Flag_INT,Llave,Semana,Tipo_Asignacion
236597,PENDIENTE,1-3I1CG8PW,1-274355953700,RANCAGUA,"2026-01-31 00:00:00,000",13:00:00-16:00:00,JUG5,Triage Internet,5443033-7,RANC020004,"2026-01-29 12:40:01,000",2026-01,0,5443033-7_RANC020004,S05,Internet


In [11]:
recoVTR = recoVTR[['NMRO_ORDEN', 'FECHA_GESTION', 'Resultado Final Orden','MARCA','rut_cliente']]

In [12]:
recoCLARO['MARCA'] = 'CLARO'

In [13]:
# =========================
# JOIN PARA AGREGAR rut_cliente
# =========================
recoCLARO = recoCLARO.merge(
    ASIGCLARO[['numero_solicitud', 'rut_cliente']],
    left_on='FOLIO',
    right_on='numero_solicitud',
    how='left'
)

In [14]:
# =========================
# SELECCIONAR CAMPOS
# =========================
recoCLARO = recoCLARO[['FOLIO', 'FECHA_GESTION', 'Resultado Final Orden','MARCA','rut_cliente']]

# =========================
# RENOMBRAR COLUMNA
# =========================
recoCLARO = recoCLARO.rename(columns={'FOLIO': 'NMRO_ORDEN'})

# =========================
# CONCATENAR CON recoVTR
# =========================
base_reco = pd.concat([recoCLARO, recoVTR], ignore_index=True)

In [15]:
base_reco

,NMRO_ORDEN,FECHA_GESTION,Resultado Final Orden,MARCA,rut_cliente
0,1-249805032757,"2025-01-04 19:41:23,000",Mantiene Visita,CLARO,7397895-5
1,1-249901713679,"2025-01-02 20:39:33,000",Mantiene Visita,CLARO,13251456-9
2,1-249909541650,"2025-01-02 20:57:27,000",Mantiene Visita,CLARO,8783463-8
3,1-250053809524,"2025-01-05 16:26:46,000",Mantiene Visita,CLARO,17079662-4
4,1-250102492862,"2025-01-06 14:35:19,000",Mantiene Visita,CLARO,7016803-0
...,...,...,...,...,...
581597,1-276085239425,"2026-03-10 21:07:38,000",Mantiene Visita,VTR,6447145-7
581598,1-276085409505,"2026-03-10 19:19:40,000",Mantiene Visita,VTR,10137877-2
581599,1-276085603617,"2026-03-10 19:24:52,000",Mantiene Visita,VTR,6654063-4
581600,1-276085644223,"2026-03-10 18:37:54,000",Mantiene Visita,VTR,14470169-0


In [27]:
# Definir la ruta del archivo correctamente
archivo = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\Blip Power Bi Base\f_blip_ha.csv"
blip_ha = pd.read_csv(archivo, delimiter=',', encoding='latin1')  # Prueba con 'latin1'

# Ajustar las opciones para mostrar todas las columnas
pd.set_option('display.max_columns', None)

C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\3964556920.py:3: DtypeWarning: Columns (17,21,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,73,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,119,120,121,122,123) have mixed types. Specify dtype option on import or set low_memory=False.
  blip_ha = pd.read_csv(archivo, delimiter=',', encoding='latin1')  # Prueba con 'latin1'


In [28]:
# =========================
# DEJAR SOLO CAMPOS NECESARIOS
# =========================
blip_ha = blip_ha[['rut', 'CustomerPhoneNumber']]

# =========================
# LIMPIAR CAMPOS
# =========================
base_reco['rut_cliente'] = base_reco['rut_cliente'].astype(str).str.strip()
blip_ha['rut'] = blip_ha['rut'].astype(str).str.strip()

# =========================
# ELIMINAR DUPLICADOS POR RUT
# =========================
blip_ha = blip_ha.drop_duplicates(subset=['rut'])

# =========================
# MERGE
# =========================
base_reco = base_reco.merge(
    blip_ha,
    left_on='rut_cliente',
    right_on='rut',
    how='left'
)

# =========================
# ELIMINAR COLUMNA AUXILIAR
# =========================
base_reco = base_reco.drop(columns=['rut'])

In [32]:
# =========================
# FILTRAR REGISTROS QUE CONTIENEN "Cancelado"
# =========================
base_reco = base_reco[
    base_reco['Resultado Final Orden'].astype(str).str.contains('Cancelado', case=False, na=False)
]

In [41]:
base_reco['CustomerPhoneNumber'] = pd.to_numeric(
    base_reco['CustomerPhoneNumber'],
    errors='coerce'
).astype('Int64')

C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\4066465236.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_reco['CustomerPhoneNumber'] = pd.to_numeric(


In [42]:
base_reco.sample(1)

,NMRO_ORDEN,FECHA_GESTION,Resultado Final Orden,MARCA,rut_cliente,CustomerPhoneNumber
275983,1-275603715418,"2026-02-27 16:49:42,000",Cancelado,VTR,5223269-4,56987686119


In [36]:
# Definir la ruta del archivo correctamente
archivo = r"C:\Users\wduran\OneDrive - CLAROCHILE\Digital Team\NPS\NPS General Evolutivo.csv"
nps = pd.read_csv(archivo, delimiter=';', encoding='latin1')  # Prueba con 'latin1'

C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\1695694192.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  nps = pd.read_csv(archivo, delimiter=';', encoding='latin1')  # Prueba con 'latin1'


In [38]:
nps.sample(1)

,Ejecutivo,Proveedor,Cola,NPS,Comentario,Resolutividad,CSAT Asesor,Comentario CSAT,SUM,Telefono Cliente,Fecha,Semana
194530,tatiana.orozcomontoya@ext.clarovtr.cl,ATENTO COLOMBIA,Comercial_Fijo,10,null,SÃÂ­,Muy buena,null,3904852.0,56948574221,01-03-2026,1.0


In [43]:
# =========================
# FORMATO TELEFONOS
# =========================
base_reco['CustomerPhoneNumber'] = pd.to_numeric(base_reco['CustomerPhoneNumber'], errors='coerce')

nps['Telefono Cliente'] = (
    nps['Telefono Cliente']
    .astype(str)
    .str.replace(r'\D','', regex=True)
)

nps['Telefono Cliente'] = pd.to_numeric(nps['Telefono Cliente'], errors='coerce')

# =========================
# FORMATO FECHAS
# =========================
base_reco['FECHA_GESTION'] = pd.to_datetime(base_reco['FECHA_GESTION'])
nps['Fecha'] = pd.to_datetime(nps['Fecha'])

C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\2750409525.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_reco['CustomerPhoneNumber'] = pd.to_numeric(base_reco['CustomerPhoneNumber'], errors='coerce')
C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\2750409525.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  base_reco['FECHA_GESTION'] = pd.to_datetime(base_reco['FECHA_GESTION'])
C:\Users\wduran\AppData\Local\Temp\ipykernel_5892\2750409525.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



In [58]:
# =========================
# FILTRAR ENCUESTAS TECNICO
# =========================
nps = nps[
    nps['Cola'].astype(str).str.contains('Tecnico', case=False, na=False)
]

In [59]:
base_nps = base_reco.merge(
    nps[['Telefono Cliente','Fecha','NPS']],
    left_on='CustomerPhoneNumber',
    right_on='Telefono Cliente',
    how='left'
)

In [60]:
base_nps['dias'] = (base_nps['Fecha'] - base_nps['FECHA_GESTION']).dt.days

In [61]:
base_nps = base_nps[
    (base_nps['dias'] >= 0) &
    (base_nps['dias'] <= 7)
]

In [62]:
base_nps = base_nps.sort_values('dias')

base_nps = base_nps.drop_duplicates(
    subset='NMRO_ORDEN',
    keep='first'
)

In [63]:
base_nps = base_nps[
    (base_nps['Fecha'].dt.year == 2026) &
    (base_nps['Fecha'].dt.month.isin([1,2,3]))
]

In [64]:
# =========================
# CONVERTIR NPS A NUMERICO
# =========================
base_nps['NPS'] = pd.to_numeric(base_nps['NPS'], errors='coerce')

In [65]:
base_nps['mes'] = base_nps['Fecha'].dt.to_period('M')

In [66]:
base_nps['tipo_nps'] = 'pasivo'

base_nps.loc[base_nps['NPS'] >= 9, 'tipo_nps'] = 'promotor'
base_nps.loc[base_nps['NPS'] <= 6, 'tipo_nps'] = 'detractor'

In [67]:
nps_mes = base_nps.groupby(['mes','tipo_nps']).size().unstack().fillna(0)

nps_mes['total'] = nps_mes.sum(axis=1)

nps_mes['NPS'] = (
    (nps_mes['promotor'] - nps_mes['detractor']) /
    nps_mes['total']
) * 100

print(nps_mes)

tipo_nps  detractor  pasivo  promotor  total        NPS
mes                                                    
2026-01         113      68       180    361  18.559557
2026-02         183      93       241    517  11.218569
2026-03          22      10        39     71  23.943662
